# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f'Total Revenue: ${total_revenue:.2f}')
print(f'Total Units: {total_units}')
print(df.head())
print(len(df))


Total Revenue: $8520.00
Total Units: 783
  vendor_id  category  qty  price  revenue
0      V-10     Drink    2   24.0     48.0
1      V-18  RainGear    1   12.0     12.0
2      V-18     Drink    3    4.5     13.5
3      V-10      Food    2   12.0     24.0
4      V-18     Drink    3    7.5     22.5
400


After finding total revenue and units, the orders brought in a total revenue of 8,520.00 dollars across all four vendors and four product categories and there were 783 total units describing the total number of individual items sold across the 400 orders.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = (
    df.groupby('category')['revenue']
      .sum()
      .reset_index()
)

by_category['share_of_total'] = (
    by_category['revenue'] / total_revenue * 100
)

by_category = by_category.sort_values(
    'revenue', ascending=False
)

by_category


,category,revenue,share_of_total
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


Food was the largest contributor to total revenue, while RainGear contributed the smallest amount among the four categories because food accounted for approximately 50.4 percent of the 8,520 total revenue while Rain Gear only accounted for 10.6 percent.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
vendor_revenue = (
    df.groupby('vendor_id')
      .agg(
          average_revenue=('revenue', 'mean'),
          order_count=('revenue', 'count')
      )
      .reset_index()
      .sort_values('average_revenue', ascending=False)
)

vendor_revenue['average_revenue'] = vendor_revenue['average_revenue'].round(2)

vendor_revenue


,vendor_id,average_revenue,order_count
0,V-01,22.60,94
3,V-18,21.75,108
1,V-05,20.58,93
2,V-10,20.31,105


V-01 has the highest average order revenue at 22.60 per order based on 94 orders, meaning, on average, each order from V-01 generated $22.60, which was higher than the averages for the other vendors

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO

merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()

merch_share = round((merch_revenue / total_revenue) * 100, 1)

print(f"Merch revenue share: {merch_share}%")

Merch revenue share: 20.8%


 20.8 percent of the revenue comes from merch which depicts that Merch is an important component of the overall revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})
original_rows = len(df)
original_revenue = df['revenue'].sum()


joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

missing_vendor = joined.loc[
    joined['vendor_name'].isna(),
    'vendor_id'
].unique()

print("Missing vendor:", missing_vendor)

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')

assert len(joined) == original_rows
assert joined['revenue'].sum() == original_revenue

print("Row count:", len(joined))
print("Revenue total:", joined['revenue'].sum())

# TODO: merge, validate, and report the unmatched vendor

Missing vendor: ['V-18']
Row count: 400
Revenue total: 8520.0


**The unmatched vendor, and what I did about it:** _..._ The unmatched vendor was V-18 which had 108 orders in the dataset showing that V-18 was a substantial vendor. I kept all the original orders and joined the vendors in reference to the vendor names created an adjusted dataframe. In this  dataframe, I created a label/variable as "Unknown vendor" so that no orders or revenue were lost and the vendor that had an N.A after the join is the unknown vendor. The left join preserved all 400 rows and the original total revenue, as confirmed by the check because it joined on vendor_id looking for the same vendor ids between the original dataframe and vendor_names.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO

pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


Each vendor made a similar total revenues however the actual breakdowns are different because for the Unknown Vendor the drinks brought in 582.00 dollars which is more than the merch revenue of 508.50 dollars but other vendors such as Rotunda Tacos expereinced the opposite which higher Merch revenue at 489.00 dollars and lower drink revenue at 298.50 dollars.  

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.







**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

A)
The vendors should focus on increasing revenue from the categories that generated the most overall sales. For example, food generated 4,293, which was the largest category, while RainGear generated only 901.50, so vendors could consider carrying more Food products and reconsider including their RainGear. Hoos Burgers had the highest average revenue per order at 22.60, based on 94 orders, while Rotunda Tacos had a lower average of 20.58 across 93 orders. Vendors could use these results to examine which products are contributing to larger orders and adjust their product arrangements for the next game.

B)

The least trustworthy answer is Q3, because the vendors have different numbers of orders. For example, V-05 had 93 orders while V-18 had 108 orders, so their average revenues are based on different sample sizes. Although V-01 had the highest average revenue at 22.60, this does not necessarily mean it would have the highest average with the same number of orders. Additionally, V-18 was unmatched in the vendor lookup, so some of the vendor information was incomplete. Calculating the average is based on various factors and can be specified with complete product and revenue information. Seeing which vendor performed can be important for allowing that same vendor to excel and influencing other vendors to change their strategies.
